In [0]:
%sql
CREATE WIDGET TEXT catalog DEFAULT "bootcamp";
CREATE WIDGET TEXT bronze_schema DEFAULT "urbanmart_bronze_panagiotakopoulos_026";
CREATE WIDGET TEXT silver_schema DEFAULT "urbanmart_silver_panagiotakopoulos_026";
CREATE WIDGET TEXT gold_schema DEFAULT "urbanmart_gold_panagiotakopoulos_026";

USE CATALOG IDENTIFIER(:catalog);
USE SCHEMA IDENTIFIER(:gold_schema);

In [0]:
%sql
-- TEST 1: bronze order-item ROWS - rejected order-item ROWS = fact rows (same grain: physical rows on both sides).
-- etl_rejects holds one row per rejected bronze row (exact-duplicate extras, conflicting keys, orphans, invalid...).
WITH c AS (
  SELECT (SELECT COUNT(*) FROM IDENTIFIER(:bronze_schema || '.order_items'))AS bronze_rows,
         (SELECT COUNT(*) FROM IDENTIFIER(:silver_schema || '.etl_rejects') WHERE source_table = 'order_items') AS reject_rows,
         (SELECT COUNT(*) FROM fact_sales)AS fact_rows
)
SELECT bronze_rows, reject_rows, fact_rows,
       assert_true(bronze_rows - reject_rows = fact_rows,
         format_string('T1 FAILED: bronze %d - rejected %d = %d, but fact_sales has %d rows',
                       bronze_rows, reject_rows, bronze_rows - reject_rows, fact_rows)) AS t1_row_reconciliation
FROM c;

bronze_rows,reject_rows,fact_rows,t1_row_reconciliation
24111,115,23996,null


In [0]:
%sql
-- TEST 1b: fact grain really is one row per order_line_id, and every silver line is in the fact
SELECT assert_true((SELECT COUNT(*) FROM fact_sales) = (SELECT COUNT(DISTINCT order_line_id) FROM fact_sales),
         'T1b FAILED: fact_sales grain broken (duplicate order_line_id)') AS grain_unique,
       assert_true((SELECT COUNT(*) FROM IDENTIFIER(:silver_schema || '.order_items')) = (SELECT COUNT(*) FROM fact_sales),
         'T1b FAILED: silver order_items rows <> fact_sales rows') AS silver_to_fact_rows;



grain_unique,silver_to_fact_rows
null,null


In [0]:
%sql
-- TEST 2: silver net_amount (same formula, computed independently) = gold net_amount
WITH s AS (
  SELECT SUM(CAST(quantity * unit_price AS DECIMAL(14,2))
           - CAST(ROUND(quantity * unit_price * discount_pct, 2) AS DECIMAL(14,2))) AS silver_net
  FROM IDENTIFIER(:silver_schema || '.order_items')
), g AS (SELECT SUM(net_amount) AS gold_net FROM fact_sales)
SELECT silver_net, gold_net,
       assert_true(silver_net = gold_net,
         format_string('T2 FAILED: silver net %s <> gold net %s', CAST(silver_net AS STRING), CAST(gold_net AS STRING))) AS t2_net_matches
FROM s CROSS JOIN g;

silver_net,gold_net,t2_net_matches
10063161.03,10063161.03,null


In [0]:
%sql
-- TEST 3: no orphan foreign keys in the fact
SELECT
  assert_true((SELECT COUNT(*) FROM fact_sales f LEFT ANTI JOIN dim_date d ON f.date_key = d.date_key) = 0, 'T3 FAILED: orphan date_key') AS fk_date,
  assert_true((SELECT COUNT(*) FROM fact_sales f LEFT ANTI JOIN dim_customer c ON f.customer_key = c.customer_key) = 0, 'T3 FAILED: orphan customer_key') AS fk_customer,
  assert_true((SELECT COUNT(*) FROM fact_sales f LEFT ANTI JOIN dim_product p ON f.product_key = p.product_key)  = 0, 'T3 FAILED: orphan product_key') AS fk_product,
  assert_true((SELECT COUNT(*) FROM fact_sales f LEFT ANTI JOIN dim_store s ON f.store_key = s.store_key) = 0, 'T3 FAILED: orphan store_key') AS fk_store;



fk_date,fk_customer,fk_product,fk_store
null,null,null,null


In [0]:
%sql
-- TEST 4: no duplicate / NULL primary keys, and no duplicate natural keys in dimensions
SELECT
  assert_true((SELECT COUNT(*) - COUNT(DISTINCT date_key) FROM dim_date) = 0 AND (SELECT COUNT_IF(date_key IS NULL) FROM dim_date) = 0, 'T4 FAILED: dim_date PK') AS pk_date,
  assert_true((SELECT COUNT(*) - COUNT(DISTINCT customer_key) FROM dim_customer) = 0 AND (SELECT COUNT_IF(customer_key IS NULL) FROM dim_customer) = 0, 'T4 FAILED: dim_customer PK') AS pk_customer,
  assert_true((SELECT COUNT(*) - COUNT(DISTINCT product_key) FROM dim_product) = 0 AND (SELECT COUNT_IF(product_key IS NULL)  FROM dim_product) = 0, 'T4 FAILED: dim_product PK') AS pk_product,
  assert_true((SELECT COUNT(*) - COUNT(DISTINCT store_key) FROM dim_store) = 0 AND (SELECT COUNT_IF(store_key IS NULL) FROM dim_store) = 0, 'T4 FAILED: dim_store PK') AS pk_store,
  assert_true((SELECT COUNT(*) - COUNT(DISTINCT order_line_id) FROM fact_sales) = 0 AND (SELECT COUNT_IF(order_line_id IS NULL) FROM fact_sales) = 0, 'T4 FAILED: fact_sales PK') AS pk_fact,
  assert_true((SELECT COUNT(*) - COUNT(DISTINCT customer_id) FROM dim_customer) = 0, 'T4 FAILED: dim_customer natural key duplicated') AS nk_customer,
  assert_true((SELECT COUNT(*) - COUNT(DISTINCT product_id) FROM dim_product) = 0, 'T4 FAILED: dim_product natural key duplicated') AS nk_product,
  assert_true((SELECT COUNT(*) - COUNT(DISTINCT store_id) FROM dim_store) = 0, 'T4 FAILED: dim_store natural key duplicated') AS nk_store;



pk_date,pk_customer,pk_product,pk_store,pk_fact,nk_customer,nk_product,nk_store
null,null,null,null,null,null,null,null


In [0]:
%sql
-- TEST 5 (extras): unknown members exist, row counts of dims reconcile with silver, measures are internally consistent
SELECT
  assert_true((SELECT COUNT(*) FROM dim_date) >= 1 AND (SELECT COUNT(*) FROM dim_date WHERE date_key = -1) = 1, 'T5 FAILED: dim_date unknown member') AS unk_date,
  assert_true((SELECT COUNT(*) FROM dim_customer  WHERE customer_key = -1) = 1, 'T5 FAILED: dim_customer unknown member') AS unk_customer,
  assert_true((SELECT COUNT(*) FROM dim_product WHERE product_key = -1) = 1, 'T5 FAILED: dim_product unknown member')  AS unk_product,
  assert_true((SELECT COUNT(*) FROM dim_store WHERE store_key = -1) = 1, 'T5 FAILED: dim_store unknown member') AS unk_store,
  assert_true((SELECT COUNT(*) FROM dim_customer) - 1 = (SELECT COUNT(*) FROM IDENTIFIER(:silver_schema || '.customers')), 'T5 FAILED: dim_customer rows <> silver customers') AS cnt_customer,
  assert_true((SELECT COUNT(*) FROM dim_product) - 1 = (SELECT COUNT(*) FROM IDENTIFIER(:silver_schema || '.products')),  'T5 FAILED: dim_product rows <> silver products')   AS cnt_product,
  assert_true((SELECT COUNT(*) FROM dim_store) - 1 = (SELECT COUNT(*) FROM IDENTIFIER(:silver_schema || '.stores')), 'T5 FAILED: dim_store rows <> silver stores') AS cnt_store,
  assert_true((SELECT SUM(margin_amount) FROM fact_sales) = (SELECT SUM(net_amount) - SUM(cost_amount) FROM fact_sales), 'T5 FAILED: margin <> net - cost') AS margin_consistent;



unk_date,unk_customer,unk_product,unk_store,cnt_customer,cnt_product,cnt_store,margin_consistent
null,null,null,null,null,null,null,null
